In [18]:
import numpy as np
import pandas as pd

from PyFairnessAI.model_selection import RandomizedSearchCVFairness
from PyFairnessAI.data import privileged_groups_sens, unprivileged_groups_sens
from PyFairnessAI.preprocessing import ReweighingMetaEstimator

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.neighbors import KNeighborsClassifier

from aif360.datasets import GermanDataset

from PyMachineLearning.models import LogisticRegressionThreshold
from PyMachineLearning.preprocessing import Encoder, Scaler, Imputer, ToPandas

pd.set_option('display.max_colwidth', None)

In [2]:
german_data_aif = GermanDataset(
        protected_attribute_names=['age'],            
        privileged_classes=[lambda x: x >= 25],      
        features_to_drop=['personal_status', 'sex'] 
    )

# Cambiar los labels de la respuesta de 2 a 0 (label 2 es desfavorable y 1 es favorable) 
german_data_aif.labels[german_data_aif.labels.ravel() == 2] =  german_data_aif.labels[german_data_aif.labels.ravel() == 2] - 2
german_data_aif.unfavorable_label = german_data_aif.unfavorable_label - 2

response_name = german_data_aif.label_names[0]
response_favorable_label = german_data_aif.favorable_label 
response_unfavorable_label = german_data_aif.unfavorable_label 
sens_variable_name = german_data_aif.protected_attribute_names[0]
sens_privileged_groups = [privileged_groups_sens(german_data_aif)]
sens_unprivileged_groups = [unprivileged_groups_sens(german_data_aif)]

print('response_name:', response_name)
print('response_favorable_label:', response_favorable_label)
print('response_unfavorable_label:', response_unfavorable_label)
print('sens_variable_name:', sens_variable_name)
print('sens_privileged_groups:', sens_privileged_groups)
print('sens_unprivileged_groups:', sens_unprivileged_groups)

response_name: credit
response_favorable_label: 1.0
response_unfavorable_label: 0.0
sens_variable_name: age
sens_privileged_groups: [{'age': 1}]
sens_unprivileged_groups: [{'age': 0}]


In [3]:
data_arr = np.concatenate([german_data_aif.labels, german_data_aif.features], axis=1)
data_col_names = [response_name] + german_data_aif.feature_names

german_data_pd = pd.DataFrame(data=data_arr, columns=data_col_names)
german_data_pd.head(6)

,credit,month,credit_amount,investment_as_income_percentage,residence_since,age,number_of_credits,people_liable_for,status=A11,status=A12,...,housing=A152,housing=A153,skill_level=A171,skill_level=A172,skill_level=A173,skill_level=A174,telephone=A191,telephone=A192,foreign_worker=A201,foreign_worker=A202
0,1.0,6.0,1169.0,4.0,4.0,1.0,2.0,1.0,1.0,0.0,...,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0
1,0.0,48.0,5951.0,2.0,2.0,0.0,1.0,1.0,0.0,1.0,...,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0
2,1.0,12.0,2096.0,2.0,3.0,1.0,1.0,2.0,0.0,0.0,...,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0
3,1.0,42.0,7882.0,2.0,4.0,1.0,1.0,2.0,1.0,0.0,...,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0
4,0.0,24.0,4870.0,3.0,4.0,1.0,2.0,2.0,1.0,0.0,...,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0
5,1.0,36.0,9055.0,2.0,4.0,1.0,1.0,2.0,0.0,0.0,...,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0


In [4]:
quant_predictors = ['month', 'credit_amount', 'investment_as_income_percentage',
                    'residence_since', 'number_of_credits', 'people_liable_for']
cat_predictors = [x for x in german_data_pd.columns if x not in quant_predictors + [response_name]]
predictors = quant_predictors + cat_predictors

In [5]:
Y = german_data_pd[response_name]
X = german_data_pd[german_data_aif.feature_names] # X = german_data_pd[quant_predictors + cat_predictors]
A = german_data_pd[sens_variable_name] # sensitive variable / protected attribute

In [6]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, train_size=0.85, random_state=123, stratify=Y)
inner = StratifiedKFold(n_splits=5, shuffle=True, random_state=111)

In [7]:
quant_pipeline = Pipeline([
    ('imputer', Imputer()),
    ('scaler', Scaler())
    ])

cat_pipeline = Pipeline([
    ('imputer', Imputer()),
    ('encoder', Encoder())
    ])

quant_cat_processing = ColumnTransformer(transformers=[('quant', quant_pipeline, quant_predictors),
                                                       ('cat', cat_pipeline, cat_predictors)])

models, pipelines, param_grid = {}, {}, {}

models['knn'] = KNeighborsClassifier()
models['logistic_reg'] = LogisticRegressionThreshold(solver='liblinear', random_state=123)
models['logistic_reg_reweighing'] = ReweighingMetaEstimator(estimator=models['logistic_reg'], prot_attr=sens_variable_name) # Fairness Pre-Processor

for key, model in models.items():

    if key == 'logistic_reg_reweighing':

        pipelines[key] = Pipeline([
                ('preprocessing', quant_cat_processing),
                ('to_pandas', ToPandas(columns=predictors)), # ReweighingMetaEstimator need a Pandas df X as input to read the sens_variable_name 
                (key, model) 
                ])            

    else:

        pipelines[key] = Pipeline([
                ('preprocessing', quant_cat_processing),
                (key, model) 
                ])

In [8]:
preprocessing_grid = {'preprocessing__quant__scaler__apply': [True, False],
                      'preprocessing__quant__scaler__method': ['standard', 'min-max'],
                      'preprocessing__cat__encoder__method': ['ordinal', 'one-hot'],
                      'preprocessing__cat__imputer__apply':  [False],
                      'preprocessing__quant__imputer__apply': [False]
                    }

In [9]:
param_grid['logistic_reg'] = preprocessing_grid.copy()
param_grid['logistic_reg'].update({'logistic_reg__penalty': ['l1', 'l2'],
                                   'logistic_reg__C':  [0.01, 0.1, 1, 10, 30, 50, 75, 100],
                                   'logistic_reg__class_weight': ['balanced', None],
                                   'logistic_reg__threshold': [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
                                  })

In [10]:
param_grid['logistic_reg_reweighing'] = preprocessing_grid.copy()
param_grid['logistic_reg_reweighing'].update({'logistic_reg_reweighing__estimator__penalty': ['l1', 'l2'],
                                              'logistic_reg_reweighing__estimator__C':  [0.01, 0.1, 1, 10, 30, 50, 75, 100],
                                              'logistic_reg_reweighing__estimator__class_weight': ['balanced', None],
                                              'logistic_reg_reweighing__estimator__threshold': [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
                                            })

In [11]:
param_grid['knn'] = preprocessing_grid.copy()
param_grid['knn'].update({'knn__n_neighbors': np.arange(1,20),
                          'knn__metric': ['cosine', 'minkowski', 'cityblock'],
                          'knn__p': np.arange(1,4)
                        })

In [12]:
fairness_random_search = RandomizedSearchCVFairness(estimator=pipelines['knn'], 
                                                    param_distributions=param_grid['knn'], 
                                                    fairness_scoring='abs_statistical_parity_difference', 
                                                    predictive_scoring='balanced_accuracy',
                                                    objective='combined', 
                                                    fairness_scoring_direction='minimize',
                                                    predictive_scoring_direction='maximize',
                                                    fairness_weight=0.5, predictive_weight=0.5,
                                                    cv=inner, n_iter=20, random_state=123,
                                                    sens_variable_name=sens_variable_name, 
                                                    priv_group=sens_privileged_groups[0][sens_variable_name],
                                                    pos_label=response_favorable_label)

fairness_random_search.fit(X=X_train, y=Y_train)

In [13]:
fairness_random_search.cv_results_.head()

,params,predictive-score,fairness-score,combined-score
18,"{'preprocessing__quant__scaler__apply': True, 'preprocessing__quant__scaler__method': 'min-max', 'preprocessing__cat__encoder__method': 'one-hot', 'preprocessing__cat__imputer__apply': False, 'preprocessing__quant__imputer__apply': False, 'knn__n_neighbors': 14, 'knn__metric': 'cosine', 'knn__p': 1}",0.641176,0.146023,0.630722
8,"{'preprocessing__quant__scaler__apply': False, 'preprocessing__quant__scaler__method': 'min-max', 'preprocessing__cat__encoder__method': 'ordinal', 'preprocessing__cat__imputer__apply': False, 'preprocessing__quant__imputer__apply': False, 'knn__n_neighbors': 12, 'knn__metric': 'cosine', 'knn__p': 2}",0.606443,0.099488,0.618568
1,"{'preprocessing__quant__scaler__apply': False, 'preprocessing__quant__scaler__method': 'min-max', 'preprocessing__cat__encoder__method': 'ordinal', 'preprocessing__cat__imputer__apply': False, 'preprocessing__quant__imputer__apply': False, 'knn__n_neighbors': 11, 'knn__metric': 'cityblock', 'knn__p': 2}",0.559944,0.035553,0.606306
15,"{'preprocessing__quant__scaler__apply': True, 'preprocessing__quant__scaler__method': 'min-max', 'preprocessing__cat__encoder__method': 'ordinal', 'preprocessing__cat__imputer__apply': False, 'preprocessing__quant__imputer__apply': False, 'knn__n_neighbors': 17, 'knn__metric': 'minkowski', 'knn__p': 3}",0.610084,0.110433,0.604998
16,"{'preprocessing__quant__scaler__apply': False, 'preprocessing__quant__scaler__method': 'standard', 'preprocessing__cat__encoder__method': 'one-hot', 'preprocessing__cat__imputer__apply': False, 'preprocessing__quant__imputer__apply': False, 'knn__n_neighbors': 5, 'knn__metric': 'cityblock', 'knn__p': 2}",0.559104,0.035614,0.603108


In [14]:
fairness_random_search = RandomizedSearchCVFairness(estimator=pipelines['logistic_reg'], 
                                                    param_distributions=param_grid['logistic_reg'], 
                                                    fairness_scoring='abs_statistical_parity_difference', 
                                                    predictive_scoring='balanced_accuracy',
                                                    objective='combined', 
                                                    fairness_scoring_direction='minimize',
                                                    predictive_scoring_direction='maximize',
                                                    fairness_weight=0.5, predictive_weight=0.5,
                                                    cv=inner, n_iter=20, random_state=123,
                                                    sens_variable_name=sens_variable_name, 
                                                    priv_group=sens_privileged_groups[0][sens_variable_name],
                                                    pos_label=response_favorable_label)

fairness_random_search.fit(X=X_train, y=Y_train)

In [15]:
fairness_random_search.cv_results_.head()

,params,predictive-score,fairness-score,combined-score
13,"{'preprocessing__quant__scaler__apply': True, 'preprocessing__quant__scaler__method': 'min-max', 'preprocessing__cat__encoder__method': 'one-hot', 'preprocessing__cat__imputer__apply': False, 'preprocessing__quant__imputer__apply': False, 'logistic_reg__penalty': 'l2', 'logistic_reg__C': 50, 'logistic_reg__class_weight': 'balanced', 'logistic_reg__threshold': 0.7}",0.721849,0.253459,0.512207
14,"{'preprocessing__quant__scaler__apply': True, 'preprocessing__quant__scaler__method': 'standard', 'preprocessing__cat__encoder__method': 'one-hot', 'preprocessing__cat__imputer__apply': False, 'preprocessing__quant__imputer__apply': False, 'logistic_reg__penalty': 'l1', 'logistic_reg__C': 100, 'logistic_reg__class_weight': 'balanced', 'logistic_reg__threshold': 0.7}",0.719888,0.254819,0.505552
15,"{'preprocessing__quant__scaler__apply': False, 'preprocessing__quant__scaler__method': 'min-max', 'preprocessing__cat__encoder__method': 'one-hot', 'preprocessing__cat__imputer__apply': False, 'preprocessing__quant__imputer__apply': False, 'logistic_reg__penalty': 'l1', 'logistic_reg__C': 50, 'logistic_reg__class_weight': 'balanced', 'logistic_reg__threshold': 0.7}",0.720728,0.256170,0.504886
0,"{'preprocessing__quant__scaler__apply': True, 'preprocessing__quant__scaler__method': 'min-max', 'preprocessing__cat__encoder__method': 'ordinal', 'preprocessing__cat__imputer__apply': False, 'preprocessing__quant__imputer__apply': False, 'logistic_reg__penalty': 'l1', 'logistic_reg__C': 0.01, 'logistic_reg__class_weight': None, 'logistic_reg__threshold': 0.7}",0.500000,0.000000,0.500000
17,"{'preprocessing__quant__scaler__apply': False, 'preprocessing__quant__scaler__method': 'min-max', 'preprocessing__cat__encoder__method': 'ordinal', 'preprocessing__cat__imputer__apply': False, 'preprocessing__quant__imputer__apply': False, 'logistic_reg__penalty': 'l1', 'logistic_reg__C': 0.01, 'logistic_reg__class_weight': 'balanced', 'logistic_reg__threshold': 0.6}",0.500000,0.000000,0.500000


In [16]:
fairness_random_search = RandomizedSearchCVFairness(estimator=pipelines['logistic_reg_reweighing'], 
                                                    param_distributions=param_grid['logistic_reg_reweighing'], 
                                                    fairness_scoring='abs_statistical_parity_difference', 
                                                    predictive_scoring='balanced_accuracy',
                                                    objective='combined', 
                                                    fairness_scoring_direction='minimize',
                                                    predictive_scoring_direction='maximize',
                                                    fairness_weight=0.5, predictive_weight=0.5,
                                                    cv=inner, n_iter=20, random_state=123,
                                                    sens_variable_name=sens_variable_name, 
                                                    priv_group=sens_privileged_groups[0][sens_variable_name],
                                                    pos_label=response_favorable_label)

fairness_random_search.fit(X=X_train, y=Y_train)

In [17]:
fairness_random_search.cv_results_.head()

,params,predictive-score,fairness-score,combined-score
2,"{'preprocessing__quant__scaler__apply': False, 'preprocessing__quant__scaler__method': 'standard', 'preprocessing__cat__encoder__method': 'one-hot', 'preprocessing__cat__imputer__apply': False, 'preprocessing__quant__imputer__apply': False, 'logistic_reg_reweighing__estimator__penalty': 'l2', 'logistic_reg_reweighing__estimator__C': 100, 'logistic_reg_reweighing__estimator__class_weight': 'balanced', 'logistic_reg_reweighing__estimator__threshold': 0.2}",0.647899,0.040397,0.710908
5,"{'preprocessing__quant__scaler__apply': False, 'preprocessing__quant__scaler__method': 'min-max', 'preprocessing__cat__encoder__method': 'one-hot', 'preprocessing__cat__imputer__apply': False, 'preprocessing__quant__imputer__apply': False, 'logistic_reg_reweighing__estimator__penalty': 'l1', 'logistic_reg_reweighing__estimator__C': 100, 'logistic_reg_reweighing__estimator__class_weight': None, 'logistic_reg_reweighing__estimator__threshold': 0.4}",0.652661,0.052784,0.682574
8,"{'preprocessing__quant__scaler__apply': True, 'preprocessing__quant__scaler__method': 'min-max', 'preprocessing__cat__encoder__method': 'ordinal', 'preprocessing__cat__imputer__apply': False, 'preprocessing__quant__imputer__apply': False, 'logistic_reg_reweighing__estimator__penalty': 'l2', 'logistic_reg_reweighing__estimator__C': 100, 'logistic_reg_reweighing__estimator__class_weight': 'balanced', 'logistic_reg_reweighing__estimator__threshold': 0.2}",0.644818,0.047436,0.681546
9,"{'preprocessing__quant__scaler__apply': True, 'preprocessing__quant__scaler__method': 'min-max', 'preprocessing__cat__encoder__method': 'ordinal', 'preprocessing__cat__imputer__apply': False, 'preprocessing__quant__imputer__apply': False, 'logistic_reg_reweighing__estimator__penalty': 'l2', 'logistic_reg_reweighing__estimator__C': 30, 'logistic_reg_reweighing__estimator__class_weight': None, 'logistic_reg_reweighing__estimator__threshold': 0.8}",0.712325,0.107112,0.647161
19,"{'preprocessing__quant__scaler__apply': False, 'preprocessing__quant__scaler__method': 'standard', 'preprocessing__cat__encoder__method': 'one-hot', 'preprocessing__cat__imputer__apply': False, 'preprocessing__quant__imputer__apply': False, 'logistic_reg_reweighing__estimator__penalty': 'l2', 'logistic_reg_reweighing__estimator__C': 10, 'logistic_reg_reweighing__estimator__class_weight': 'balanced', 'logistic_reg_reweighing__estimator__threshold': 0.2}",0.640896,0.056699,0.643212
